# Sternhaufen von Hand finden

Ein **Sternhaufen** ist eine Gruppe von Sternen, die gemeinsam aus derselben Gaswolke entstanden
ist. Solche Sterne sind deshalb bis heute *ungefähr gleich weit* von uns entfernt, stehen *nah
beieinander am Himmel* und *bewegen sich gemeinsam* durch die Galaxie.

Genau diese drei Eigenschaften nutzt du in diesem Notebook, um einen Haufen zu finden — von
Hand, nur mit deinen Augen und ein paar Zahlen. Im zweiten Notebook übernimmt dann ein
Algorithmus dieselbe Aufgabe, und du kannst vergleichen.

Wichtig dabei: Du suchst den Haufen **direkt in dem, was *Gaia* gemessen hat** — Entfernung,
Ort am Himmel, Eigenbewegung. Erst ganz am Ende, wenn deine Auswahl steht, rechnest du sie in
Positionen und Geschwindigkeiten im Raum um, um sie in 3D anzusehen.

**So gehst du vor:** ansehen → auswählen → noch einmal ansehen, ob die Auswahl passt →
Auswahl verbessern. Diese Schleife wiederholst du für die Entfernung, den Ort am Himmel und
die Eigenbewegung und kombinierst am Ende alles.

> Es gibt hier keine einzig richtige Lösung. Ziel ist, ein Gefühl dafür zu bekommen, was man
> in welcher Darstellung erkennen kann.

## Vorbereitung

Zuerst die Funktionen, die du brauchst.  `%matplotlib widget` sorgt dafür, dass die
Darstellungen später interaktiv sind — du kannst sie mit der Maus drehen und zoomen.

In [ ]:
%matplotlib widget

from stellar_cluster_finder import (
    add_distance,
    convert_to_galactic,
    load_fits,
    load_parquet,
    plot_3d,
    plot_and_save,
    plot_histogram,
    save_parquet,
    select_ellipse,
    select_range,
)

Jede Darstellung erzeugt ein **neues** Bild — du musst nichts aufräumen, damit das nächste
richtig aussieht. Weil du hier aber viel wiederholst, sammeln sich die Bilder mit der Zeit an.
Ab 20 offenen Bildern warnt matplotlib. Dann räumst du so auf:

```python
import matplotlib.pyplot as plt

plt.close("all")
```

## 1. Die Daten öffnen

Lege deine bei *Gaia* heruntergeladene Datei in den Ordner `data/` (eine Ebene über diesem
Notebook) und trage unten den Dateinamen ein. Beim ersten Mal liest du die FITS-Datei ein und
speicherst sie als Parquet-Datei — dieses Format lädt viel schneller.

In [ ]:
# Einmalig: die FITS-Datei lesen und als Parquet-Datei ablegen.
sterne = load_fits("../data/pleiades-query_0209.fits")
save_parquet(sterne, "../data/pleiades.parquet")

# Ab jetzt reicht diese Zeile -- sie lädt deutlich schneller.
sterne = load_parquet("../data/pleiades.parquet")

print(f"{len(sterne)} Sterne geladen")
sterne.head()

### Was in der Tabelle steht

Diese sechs Spalten braucht dieses Notebook. Alle sind **Messwerte** von Gaia — nichts davon
ist umgerechnet:

| Spalte | Bedeutung | Einheit |
| --- | --- | --- |
| `ra`, `dec` | Ort am Himmel (Rektaszension und Deklination) | Grad |
| `parallax` | scheinbares jährliches Hin- und Herwandern des Sterns | mas |
| `pmra`, `pmdec` | **Eigenbewegung**: wie schnell der Stern über den Himmel wandert | mas/Jahr |
| `radial_velocity` | Geschwindigkeit auf uns zu oder von uns weg | km/s |

`mas` heißt Millibogensekunde und ist ein sehr kleiner Winkel: 1 mas ist ungefähr die Breite
eines Haares, gesehen aus 10 km Entfernung.

## 2. Auswahl über die Entfernung (1D)

Die **Parallaxe** ist das Wichtigste an den Gaia-Daten: Weil sich die Erde um die Sonne bewegt,
scheinen nahe Sterne im Laufe eines Jahres leicht hin und her zu wandern. Je größer diese
Bewegung, desto näher der Stern. Daraus folgt die Entfernung als `1000 / Parallaxe` in Parsec
(1 pc ≈ 3,26 Lichtjahre).

`add_distance` macht genau diese eine Division und legt die Spalte `distance_pc` an. Umgerechnet
wird sonst noch nichts — du bleibst in den Rohdaten.

In [ ]:
sterne = add_distance(sterne)
sterne[["parallax", "distance_pc"]].head()

Sterne eines Haufens sind ungefähr gleich weit entfernt. Im Histogramm zeigt sich das als
**Spitze**, die aus dem breiten Untergrund der übrigen Sterne herausragt.

In [ ]:
plot_histogram(
    sterne,
    "distance_pc",
    bins=100,
    title="Wie weit sind die Sterne entfernt?",
    xlabel="Entfernung (pc)",
    ylabel="Anzahl Sterne",
)

Zwei Dinge erschweren den Blick auf die Spitze. Zum einen wird der Untergrund nach rechts hin
immer höher: Je weiter außen eine Kugelschale um uns liegt, desto mehr Platz hat sie, und desto
mehr Sterne stehen darin. Zum anderen ist die Spitze schmal — bei 100 Balken über den ganzen
Bereich fällt sie in nur wenige davon.

Mit `xlim` zoomst du in den Bereich, der dich interessiert. Die Balken werden dabei mitgezoomt,
du siehst also wirklich mehr Details und nicht nur einen Ausschnitt.

In [ ]:
plot_histogram(
    sterne,
    "distance_pc",
    bins=60,
    title="Entfernungen, herangezoomt",
    xlabel="Entfernung (pc)",
    ylabel="Anzahl Sterne",
    xlim=(50, 400),  # <-- anpassen, bis du die Spitze gut siehst
)

**Deine Aufgabe:** Suche die Spitze und lies ab, zwischen welchen beiden Entfernungen sie liegt.
Trage diese beiden Zahlen unten ein.

In [ ]:
# Startwerte: die mittlere Hälfte aller Sterne. Das ist nur ein Anfang -- ersetze die
# beiden Zahlen durch die Werte, die du am Rand deiner Spitze abgelesen hast.
entfernung_min = float(sterne["distance_pc"].quantile(0.25))  # <-- anpassen
entfernung_max = float(sterne["distance_pc"].quantile(0.75))  # <-- anpassen

print(f"ausgewählter Bereich: {entfernung_min:.0f} bis {entfernung_max:.0f} pc")

select_range(
    sterne,
    "distance_pc",
    minimum=entfernung_min,
    maximum=entfernung_max,
    selection_column="auswahl_entfernung",
)

Jetzt **sieh dir dasselbe Histogramm noch einmal an**, diesmal eingefärbt nach deiner Auswahl.
So siehst du sofort, ob du die Spitze getroffen hast oder zu viel bzw. zu wenig erwischt hast.

In [ ]:
plot_histogram(
    sterne,
    "distance_pc",
    color_column="auswahl_entfernung",
    bins=60,
    title="Meine Auswahl über die Entfernung",
    xlabel="Entfernung (pc)",
    ylabel="Anzahl Sterne",
    xlim=(120, 150),  # <-- derselbe Ausschnitt wie oben
)

> **Wiederhole das ruhig mehrmals.** Gehe zurück zur Zelle mit `entfernung_min` und
> `entfernung_max`, ändere die Werte und führe die beiden Zellen erneut aus, bis die Auswahl
> gut zur Spitze passt.

## 3. Auswahl über den Ort am Himmel (2D)

Ein Haufen steht auch am Himmel dicht beieinander — er ist ja eine Gruppe an *einer* Stelle.
Trage dazu `ra` gegen `dec` auf. Das ist genau die Ansicht, die du auch am Teleskop hättest.

Bei so vielen Sternen zeichnet die Funktion die Punkte von sich aus durchsichtig und schreibt
die gewählte Transparenz in die Ausgabe: Deckende Punkte verdecken einander und machen aus dem
ganzen Feld eine gleichmäßig ausgefüllte Fläche, während sich durchsichtige Punkte dort
aufaddieren, wo die Sterne dicht stehen — und genau diese dunklere Stelle suchst du.

In [ ]:
plot_and_save(
    sterne,
    "ra",
    "dec",
    title="Wo stehen die Sterne am Himmel?",
    xlabel="Rektaszension (Grad)",
    ylabel="Deklination (Grad)",
    alpha=0.1,  # <-- einkommentieren: ohne Transparenz, zum Vergleich
)

**Deine Aufgabe:** Suche die dichteste Stelle und lege eine Ellipse darum. Die Ellipse
beschreibst du mit fünf Zahlen:

| Zahl | Bedeutung |
| --- | --- |
| `x0`, `y0` | Mittelpunkt der Ellipse |
| `width` | Breite (ganze Achse, nicht halbe) |
| `height` | Höhe |
| `angle` | Drehung gegen den Uhrzeigersinn in Grad |

Zeichne sie zuerst nur ein, ohne schon auszuwählen — so kannst du die Zahlen in Ruhe anpassen.

In [ ]:
# Startwert: eine weite Ellipse in der Mitte deiner Daten. Verschiebe und verkleinere sie,
# bis sie um die dichteste Stelle liegt.
ellipse_himmel = {
    "x0": float(sterne["ra"].median()),  # <-- anpassen
    "y0": float(sterne["dec"].median()),  # <-- anpassen
    "width": 2 * float(sterne["ra"].std()),  # <-- anpassen
    "height": 2 * float(sterne["dec"].std()),  # <-- anpassen
    "angle": 0.0,  # <-- anpassen
}

plot_and_save(
    sterne,
    "ra",
    "dec",
    title="Passt die Ellipse?",
    xlabel="Rektaszension (Grad)",
    ylabel="Deklination (Grad)",
    ellipse_params=ellipse_himmel,
)

Wenn die Ellipse sitzt, wählst du mit **denselben** Zahlen die Sterne darin aus.

In [ ]:
select_ellipse(sterne, "ra", "dec", ellipse_himmel, selection_column="auswahl_himmel")

Und **noch einmal ansehen** — jetzt zeigen Farbe *und* Form der Punkte, was du ausgewählt hast.

In [ ]:
plot_and_save(
    sterne,
    "ra",
    "dec",
    title="Meine Auswahl am Himmel",
    xlabel="Rektaszension (Grad)",
    ylabel="Deklination (Grad)",
    color_col="auswahl_himmel",
    ellipse_params=ellipse_himmel,
)

> **Auch hier: wiederholen.** Ändere die Zahlen in `ellipse_himmel` und führe die Zellen erneut
> aus. Wenn `0 von ...` ausgewählt wurden, liegt deine Ellipse neben den Daten — vergleiche
> ihren Mittelpunkt mit den Achsenbeschriftungen im Bild.

## 4. Auswahl über die Eigenbewegung (2D)

Das stärkste Merkmal kommt zum Schluss. Alle Sterne wandern langsam über den Himmel; wie
schnell und in welche Richtung, steht in `pmra` und `pmdec` — der **Eigenbewegung**.

Die Mitglieder eines Haufens sind gemeinsam entstanden und fliegen deshalb bis heute
gemeinsam durch die Galaxie. Am Himmel wandern sie dadurch alle in dieselbe Richtung und
gleich schnell und bilden in dieser Darstellung einen dichten Klumpen. Zufällig benachbarte
Sterne tun das nicht: Sie sind über das ganze Bild verteilt. Deshalb ist ein Haufen hier oft
viel deutlicher zu sehen als am Himmel selbst.

Ein Hinweis vorweg: Einige wenige sehr schnelle Sterne ziehen beide Achsen weit auseinander,
sodass der Klumpen zu einem Punkt in der Mitte zusammenschrumpft. Mit `xlim` und `ylim`
schneidest du beide Achsen zu und holst ihn dir heran. An der Tabelle ändert das nichts, nur am
Ausschnitt. Ein guter erster Schnitt lässt auf jeder Achse das äußerste Prozent weg; danach
liest du engere Zahlen an den Achsen ab und setzt sie von Hand ein.

In [ ]:
plot_and_save(
    sterne,
    "pmra",
    "pmdec",
    title="Wie wandern die Sterne über den Himmel?",
    xlabel="Eigenbewegung in RA (mas/Jahr)",
    ylabel="Eigenbewegung in Dec (mas/Jahr)",
    alpha=0.01,
    # Die äußersten 1% je Achse weglassen — das sind die wenigen sehr schnellen Sterne, die
    # das Bild auseinanderziehen. Danach engere Zahlen von Hand einsetzen.
    # xlim=(float(sterne["pmra"].quantile(0.01)), float(sterne["pmra"].quantile(0.99))),
    # ylim=(float(sterne["pmdec"].quantile(0.01)), float(sterne["pmdec"].quantile(0.99))),
)

In [ ]:
ellipse_eigenbewegung = {
    "x0": float(sterne["pmra"].median()),  # <-- anpassen
    "y0": float(sterne["pmdec"].median()),  # <-- anpassen
    "width": 2 * float(sterne["pmra"].std()),  # <-- anpassen
    "height": 2 * float(sterne["pmdec"].std()),  # <-- anpassen
    "angle": 0.0,  # <-- anpassen
}

plot_and_save(
    sterne,
    "pmra",
    "pmdec",
    title="Passt die Ellipse?",
    xlabel="Eigenbewegung in RA (mas/Jahr)",
    ylabel="Eigenbewegung in Dec (mas/Jahr)",
    ellipse_params=ellipse_eigenbewegung,
    # Derselbe Ausschnitt wie oben, dann siehst du die Ellipse genauer.
    # xlim=(float(sterne["pmra"].quantile(0.01)), float(sterne["pmra"].quantile(0.99))),
    # ylim=(float(sterne["pmdec"].quantile(0.01)), float(sterne["pmdec"].quantile(0.99))),
)

In [ ]:
select_ellipse(sterne, "pmra", "pmdec", ellipse_eigenbewegung, selection_column="auswahl_eigenbewegung")

plot_and_save(
    sterne,
    "pmra",
    "pmdec",
    title="Meine Auswahl über die Eigenbewegung",
    xlabel="Eigenbewegung in RA (mas/Jahr)",
    ylabel="Eigenbewegung in Dec (mas/Jahr)",
    color_col="auswahl_eigenbewegung",
    ellipse_params=ellipse_eigenbewegung,
)

## 5. Die drei Auswahlen kombinieren

Du hast jetzt drei Spalten mit `True` und `False`. Mit `&` („und") verlangst du, dass ein Stern
**alle** Bedingungen erfüllt — nur solche Sterne sind gute Haufenkandidaten.

In [ ]:
sterne["auswahl_gesamt"] = sterne["auswahl_entfernung"] & sterne["auswahl_himmel"] & sterne["auswahl_eigenbewegung"]

for spalte in ["auswahl_entfernung", "auswahl_himmel", "auswahl_eigenbewegung", "auswahl_gesamt"]:
    print(f"{spalte:24s} {sterne[spalte].sum():5d} Sterne")

**Sieh dir an, was das gebracht hat.** Trage die Eigenbewegung noch einmal auf, diesmal
eingefärbt nach der kombinierten Auswahl, und vergleiche mit dem Bild aus Schritt 4.

In [ ]:
plot_and_save(
    sterne,
    "pmra",
    "pmdec",
    title="Kombinierte Auswahl",
    xlabel="Eigenbewegung in RA (mas/Jahr)",
    ylabel="Eigenbewegung in Dec (mas/Jahr)",
    color_col="auswahl_gesamt",
)

> **Jetzt wird es interessant.** Nimm das, was du hier siehst, mit zurück zu den Schritten 2
> bis 4 und verbessere deine Auswahlen. Wenn du zum Beispiel in der Eigenbewegung einen klaren
> Klumpen siehst, kannst du die Ellipse am Himmel großzügiger machen — die Eigenbewegung
> sortiert die falschen Sterne ohnehin aus.
>
> Probiere auch `|` statt `&` („oder") aus: Wie viele Sterne erfüllen dann mindestens eine
> Bedingung? Was sagt dir der Unterschied?

## 6. Vom Messwert zum Ort im Raum

Bis hierhin hast du nur mit dem gearbeitet, was Gaia direkt gemessen hat. Das reicht, um einen
Haufen zu finden — aber es sagt dir noch nicht, **wo** die Sterne wirklich stehen und **wie
schnell** sie tatsächlich sind. Ein Stern am Rand deiner Ellipse kann doppelt so weit entfernt
sein wie einer in der Mitte.

`convert_to_galactic` rechnet deshalb aus Ort am Himmel, Entfernung, Eigenbewegung und
Radialgeschwindigkeit sechs neue Spalten aus:

| Spalten | Bedeutung | Einheit |
| --- | --- | --- |
| `X`, `Y`, `Z` | wirkliche Position im Raum | pc |
| `U`, `V`, `W` | wirkliche Geschwindigkeit durch den Raum | km/s |

Das Koordinatensystem ist an der Milchstraße ausgerichtet, mit der Sonne im Ursprung.

In [ ]:
sterne = convert_to_galactic(sterne)
sterne[["X", "Y", "Z", "U", "V", "W"]].head()

Beim Umrechnen erscheint eine Warnung: Für viele Sterne hat Gaia **keine
Radialgeschwindigkeit** gemessen — die ist viel aufwendiger zu bestimmen als die Eigenbewegung,
und Gaia schafft sie nur für die helleren Sterne. Bei diesen Sternen bleiben `U`, `V` und `W`
leer, die Position `X`, `Y`, `Z` ist aber trotzdem da.

Genau deshalb hast du zuerst in den Rohdaten ausgewählt: Deine Auswahl steht schon, bevor diese
Lücken überhaupt eine Rolle spielen. Wie groß die Lücke ist, siehst du gleich unten.

In [ ]:
print(sterne[["X", "Y", "Z", "U", "V", "W"]].isna().sum())

## 7. Der Haufen in 3D

Jetzt die Probe aufs Exempel: Bilden deine ausgewählten Sterne wirklich eine Gruppe im Raum?
Die folgende Darstellung kannst du **mit der Maus drehen** — nutze das, sie ist aus jeder
Richtung anders aussagekräftig.

Achte besonders darauf, ob deine Auswahl in der Tiefe zusammenhält. Am Himmel sah sie ja
zwangsläufig kompakt aus — das war schließlich dein Auswahlkriterium.

In [ ]:
plot_3d(
    sterne,
    "X",
    "Y",
    "Z",
    title="Meine Auswahl im Raum",
    xlabel="X (pc)",
    ylabel="Y (pc)",
    zlabel="Z (pc)",
    color_col="auswahl_gesamt",
)

Auch der Geschwindigkeitsraum hat drei Achsen — sieh ihn dir genauso an. Hier fehlen die Sterne
ohne Radialgeschwindigkeit.

In [ ]:
plot_3d(
    sterne,
    "U",
    "V",
    "W",
    title="Meine Auswahl im Geschwindigkeitsraum",
    xlabel="U (km/s)",
    ylabel="V (km/s)",
    zlabel="W (km/s)",
    color_col="auswahl_gesamt",
)

## Speichern für das nächste Notebook

In [ ]:
save_parquet(sterne, "../data/meine_auswahl.parquet")
print("gespeichert")

## Fragen zum Nachdenken

- In welcher der drei Darstellungen — Entfernung, Ort am Himmel, Eigenbewegung — war der Haufen
  am deutlichsten zu erkennen? Woran könnte das liegen?
- Wie viele Sterne hat jede einzelne Auswahl geliefert, und wie viele blieben nach dem
  Kombinieren übrig? Was sagt dieser Unterschied darüber aus, wie viele Sterne nur *zufällig*
  in einer der Auswahlen gelandet sind?
- Angenommen, zwei Sterne stehen am Himmel direkt nebeneinander, wandern aber in ganz
  verschiedene Richtungen. Gehören sie zum selben Haufen?
- Du hast in den Rohdaten ausgewählt und erst danach umgerechnet. Was wäre anders gewesen,
  wenn du zuerst umgerechnet und dann ausgewählt hättest?
- Wo bist du dir bei deiner Auswahl unsicher? Welche zusätzliche Messung würde dir helfen?

Im nächsten Notebook lässt du Algorithmen dieselbe Aufgabe lösen — und vergleichst.